In [1]:
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [2]:
from openai import OpenAI
import json

client = OpenAI()

In [7]:
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather in the given location",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "an location like Baltimore",
                },
            },
            "required": ["city"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

In [4]:
import requests

def get_weather(city : str) -> str:
    """Get current weather for a given city using Open-Meteo."""
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json",
    }

    geo_response = requests.get(geo_url, params=geo_params, timeout=10)
    geo_response.raise_for_status()
    geo_data = geo_response.json()

    if "results" not in geo_data or not geo_data["results"]:
        return f"Could not find location: {city}"
    
    location = geo_data["results"][0]
    latitude = location["latitude"]
    longitude = location["longitude"]
    resolved_name = location.get("name", city)
    country = location.get("country", "")
    admin1 = location.get("admin1", "")

    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,precipitation,weather_code,wind_speed_10m",
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch",
        "timezone": "auto",
    }
    weather_response = requests.get(weather_url, params=weather_params, timeout=10)
    weather_response.raise_for_status()
    weather_data = weather_response.json()

    current = weather_data.get("current", {})
    units = weather_data.get("current_units", {})
    temp = current.get("temperature_2m")
    humidity = current.get("relative_humidity_2m")
    precipitation = current.get("precipitation")
    wind_speed = current.get("wind_speed_10m")
    weather_code = current.get("weather_code")

    return (
        f"Current weather in {resolved_name}, {admin1}, {country}: "
        f"temperature {temp}{units.get('temperature_2m', '')}, "
        f"humidity {humidity}{units.get('relative_humidity_2m', '')}, "
        f"precipitation {precipitation}{units.get('precipitation', '')}, "
        f"wind speed {wind_speed} {units.get('wind_speed_10m', '')}, "
        f"weather code {weather_code}."
    )


In [5]:
input_list = [
    {"role": "user", "content": "What's the weather in San Francisco?"},
]

In [8]:
response = client.responses.create(
    model="gpt-5.5",
    tools=tools,
    input=input_list,
)

In [9]:
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_weather":
            city = json.loads(item.arguments)["city"]
            weather = get_weather(city)

            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": weather
            })

In [10]:
response = client.responses.create(
    model="gpt-5.5",
    instructions="Respond only with a weather generated by a tool.",
    tools=tools,
    input=input_list,
)

In [11]:
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final output:
{
  "id": "resp_02697cc5792285fa006a298a41400c81a3807d78b40c9b0342",
  "created_at": 1781107265.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with a weather generated by a tool.",
  "metadata": {},
  "model": "gpt-5.5-2026-04-23",
  "object": "response",
  "output": [
    {
      "id": "msg_02697cc5792285fa006a298a42219481a3a67cc1b947ddf42d",
      "content": [
        {
          "annotations": [],
          "text": "Current weather in San Francisco, California, United States: temperature 64.9°F, humidity 75%, precipitation 0.0inch, wind speed 0.3 mp/h, weather code 0.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_weather",
      "parameters": {
        "type": "object"